# init-process-group-nccl — ex2: @contextmanager dist_session: destroy guaranteed on exception

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `init-process-group-nccl`. Running the final beacon cell reports progress against the `Distributed: init_process_group nccl` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Distributed: init_process_group nccl` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`init-process-group-nccl`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "init-process-group-nccl"
DD_SUBTOPIC = "Distributed: init_process_group nccl"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## `init_process_group` wrapped in a context manager

Ex1 called `init_process_group` + `destroy_process_group` by hand. The footgun: if anything between init and destroy raises (loss explodes, OOM, data loader crashes), `destroy` is skipped — the rendezvous port stays bound and the NEXT training run hangs on init.

Context-manager wrap fixes it with `try/finally`:

```python
from contextlib import contextmanager

@contextmanager
def dist_session(rank, world_size, port, backend='gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist.init_process_group(backend=backend, rank=rank,
                            world_size=world_size)
    try:
        yield
    finally:
        dist.destroy_process_group()
```

**Why `finally`, not `try/except`.** A `try/except` swallows the exception. We want the exception to propagate AND destroy to run. `finally` is the only construct that guarantees both.

**Where this pattern lives in real code.** `torch.distributed.run` (the `torchrun` launcher) wraps every worker in this exact pattern internally. Hand-rolling it for notebook-launched workers gives you the same crash-resilience.

### Exercise 2 — @contextmanager dist_session: destroy guaranteed on exception

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `@contextlib.contextmanager` + `try/finally` to wrap `init_process_group` and `destroy_process_group` so destroy is guaranteed even when the body raises.
> Keywords: contextmanager, init_process_group, destroy_process_group, try-finally
> ```

**KCs targeted:** `contextmanager-init-destroy`, `destroy-runs-even-on-exception`

Implement `ex2_dist_session(rank, world_size, port, dist_module, backend='gloo')`, a context manager. Required behavior:

1. Decorate with `@contextlib.contextmanager`.
2. BEFORE the `yield`: set `os.environ['MASTER_ADDR'] = '127.0.0.1'` and `os.environ['MASTER_PORT'] = str(port)`. Call `dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)`.
3. Wrap the `yield` in `try / finally`. The `finally` block MUST call `dist_module.destroy_process_group()`, with NO conditional guards — destroy always runs, even when the body raised.
4. `yield` no value (a bare `yield`) — the context manager exists for its side effect.

Input: `rank`, `world_size`, `port` — ints; `dist_module` — torch.distributed or mock; `backend` — str, default `'gloo'`.
Yields: nothing.

The test runs the context manager (a) normally, (b) with an exception inside the `with` body. In both cases destroy must have been called exactly once.

In [ ]:
import contextlib
import os

@contextlib.contextmanager
def ex2_dist_session(rank: int, world_size: int, port: int, dist_module, backend: str = 'gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)
    try:
        yield
    finally:
        dist_module.destroy_process_group()


<details><summary>Solution</summary>

```python
import contextlib
import os

@contextlib.contextmanager
def ex2_dist_session(rank: int, world_size: int, port: int, dist_module, backend: str = 'gloo'):
    os.environ['MASTER_ADDR'] = '127.0.0.1'
    os.environ['MASTER_PORT'] = str(port)
    dist_module.init_process_group(backend=backend, rank=rank, world_size=world_size)
    try:
        yield
    finally:
        dist_module.destroy_process_group()
```

**Why `finally`, not `try/except`.** `try/except` would catch the exception and swallow it — wrong. `finally` runs the cleanup AND re-raises whatever was in flight. The two-line discipline of `init` → `try: yield ... finally: destroy` is the entire pattern.

**Env vars before init.** `init_process_group` reads `MASTER_ADDR` and `MASTER_PORT` from `os.environ` if not passed as kwargs. Setting them BEFORE the init call (not after) is mandatory.

**Real-world equivalent.** `torchrun` (the elastic launcher) wraps every worker in this exact pattern internally. For notebook-launched workers, hand-rolling the context manager gives you the same crash-resilience without launching from the CLI.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()